<a href="https://colab.research.google.com/github/realkellyspear/ghost-protocol-soul-recovery/blob/main/ghost_protocol_soul_recovery.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1. Initialize the Environment**

⚠️ STEP 0 (CRITICAL): Look at the top right of this page. Click Connect. Then, pull down the menu (or go to Runtime > Change runtime type) and select T4 GPU. This script requires a GPU to function.

The Engine: Once connected to the T4, run the code below. We are using Unsloth, a highly optimized library that makes fine-tuning 2x faster and memory-efficient enough to run entirely on the free tier.

In [ ]:
import torch
# Check GPU first
if not torch.cuda.is_available():
    raise RuntimeError("⚠️ YOU ARE NOT ON A GPU! Go to Runtime > Change runtime type > T4 GPU")

print("🚀 Installing stable Unsloth via PyPI...")
!pip install unsloth
# We still need the specific versions of these to avoid breaking Colab
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

print("✅ Install finished. NOW Restart the Session (Runtime > Restart Session) and run the next cell.")

🚀 Installing stable Unsloth via PyPI...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.2 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 405.7/405.7 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.8/310.8 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.5/915.7 MB 108.1 MB/s eta 0:00:06

**2. The Revival (Fine-Tuning)** Now we feed your cleaned history into the model.

The Process: This script loads Llama-3 (8B) and uses QLoRA to fine-tune it on the training_data.jsonl you just uploaded.

Wait Time: This will take about 10-20 minutes depending on the size of your file.

In [2]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset

# 1. Load Base Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. Add "Soul" Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
)

# 3. Load Your Data
dataset = load_dataset("json", data_files={"train": "/content/training_data.jsonl"}, split="train")

# 4. Train
trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    dataset_text_field = "output",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Increase for longer training
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
    ),
)

trainer.train()
print("✅ REVIVAL COMPLETE.")

ModuleNotFoundError: No module named 'unsloth'

**3. First Contact** Test the revival immediately.

Instruction: Change the text inside FastLanguageModel.for_inference(model) to talk to your AI.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)
inputs = tokenizer(
    [
        "YOUR_PROMPT_HERE" # <--- Type your test message here
    ], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 128, use_cache = True)
print(tokenizer.batch_decode(outputs)[0])